# vLLM Config Sweep

Measures throughput, TTFT, ITL, and latency for `Qwen/Qwen2.5-0.5B-Instruct`
under different vLLM configurations on a Colab T4.

Configs:
- fp16, max_num_seqs=8
- fp16, max_num_seqs=32
- int4 (AWQ), max_num_seqs=8
- int4 (AWQ), max_num_seqs=32

Concurrency levels per config: 1, 4, 16.

Every run writes a JSON to /content/results/ immediately. Download the
zip before the session ends — Colab will disconnect without warning.

In [ ]:
!pip install -q "vllm>=0.6.3" "openai>=1.52.2" nest_asyncio
!nvidia-smi
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6

In [ ]:
!hf download Qwen/Qwen2.5-0.5B-Instruct
!hf download Qwen/Qwen2.5-0.5B-Instruct-AWQ

Hint: A new version of huggingface_hub (2.1.1) is available! You are using version 1.31.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.92k [00:00<?, ?B/s]         

Fetching 10 files:  10% 1/10 [00:00<00:01,  8.16it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):  30% 4.92k/16.3k [00:00<00:00, 40.5kB/s]
Reconstructing (incomplete total...):   0% 4.92k/1.69M [00:00<00:41, 40.5kB/s]
Reconstructing (incomplete total...):   1% 16.3k/1.69M [00:00<00:41, 40.5kB/s]
Reconstructing (incomplete total...):   0% 17.8k/990M [00:00<6:46:53, 40.5kB/s]
Reconstructing (incomplete total...):   0% 1

In [ ]:
import json, os, subprocess, time, signal, sys
from pathlib import Path
import nest_asyncio
nest_asyncio.apply()

RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)


def start_server(model, dtype, max_num_seqs, gpu_mem_util=0.85,
                 port=8000, log_path=None):
    """Launch vLLM server as a background process. Returns Popen."""
    cmd = [
        "vllm", "serve",
        "--model", model,
        "--dtype", dtype,
        "--max-num-seqs", str(max_num_seqs),
        "--gpu-memory-utilization", str(gpu_mem_util),
        "--max-model-len", "2048",
        "--port", str(port),
    ]
    log = open(log_path, "w") if log_path else None
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    return proc


def wait_for_server(port=8000, timeout=300):
    """Poll the server's /health endpoint until ready."""
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen(
                f"http://localhost:{port}/health", timeout=2
            ):
                return True
        except Exception:
            time.sleep(2)
    return False


def stop_server(proc):
    proc.send_signal(signal.SIGINT)
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()


def run_benchmark(config_name, port=8000, num_prompts=100,
                  max_concurrency=16, input_len=128, output_len=128):
    """Invoke vLLM's benchmark CLI against a running server."""
    cmd = [
        "vllm", "bench", "serve",
        "--backend", "openai",
        "--base-url", f"http://localhost:{port}",
        "--model", "Qwen/Qwen2.5-0.5B-Instruct",
        "--num-prompts", str(num_prompts),
        "--max-concurrency", str(max_concurrency),
        "--dataset-name", "random",
        "--random-input-len", str(input_len),
        "--random-output-len", str(output_len),
        "--save-result",
        "--result-filename", str(RESULTS_DIR / f"{config_name}.json"),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=False)

In [ ]:
# Smoke test: start one server, confirm it's reachable, run one benchmark.
# If this cell fails, DO NOT proceed — debug from the server log first.
!pkill -f vllm || true
time.sleep(3)

proc = start_server(
    "Qwen/Qwen2.5-0.5B-Instruct", "float16", 8,
    log_path=str(RESULTS_DIR / "test_server.log"),
)

print("waiting for server (up to 5 min)...")
if wait_for_server(timeout=300):
    print("server up")
    !curl -s http://localhost:8000/v1/models | head -c 400
    print()
    run_benchmark("test_c1", max_concurrency=1,
                  num_prompts=10, input_len=64, output_len=64)
else:
    print("server failed to start — see /content/results/test_server.log")
    !tail -50 /content/results/test_server.log

stop_server(proc)
time.sleep(5)

^C
waiting for server (up to 5 min)...
server up
{"object":"list","data":[{"id":"Qwen/Qwen2.5-0.5B-Instruct","object":"model","created":1791105889,"owned_by":"vllm","root":"Qwen/Qwen2.5-0.5B-Instruct","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-bd7afdee85352ade","object":"model_permission","created":1791105889,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":t
vllm bench serve --backend openai --base-url http://localhost:8000 --model Qwen/Qwen2.5-0.5B-Instruct --num-prompts 10 --max-concurrency 1 --dataset-name random --random-input-len 64 --random-output-len 64 --save-result --result-filename /content/results/test_c1.json


In [ ]:
CONFIGS = [
    # (name, model, dtype, max_num_seqs)
    ("fp16_mns8",  "Qwen/Qwen2.5-0.5B-Instruct",     "float16", 8),
    ("fp16_mns32", "Qwen/Qwen2.5-0.5B-Instruct",     "float16", 32),
    ("int4_mns8",  "Qwen/Qwen2.5-0.5B-Instruct-AWQ", "float16", 8),
    ("int4_mns32", "Qwen/Qwen2.5-0.5B-Instruct-AWQ", "float16", 32),
]

CONCURRENCY_LEVELS = [1, 4, 16]

for name, model, dtype, mns in CONFIGS:
    print(f"\n=== {name} ===")
    log_path = RESULTS_DIR / f"{name}_server.log"
    proc = start_server(model, dtype, mns, log_path=str(log_path))

    if not wait_for_server(timeout=300):
        print(f"  server failed to start; see {log_path}")
        stop_server(proc)
        time.sleep(5)
        continue

    print("  server up")
    for conc in CONCURRENCY_LEVELS:
        run_name = f"{name}_c{conc}"
        print(f"  concurrency={conc}")
        run_benchmark(run_name, max_concurrency=conc)
        time.sleep(3)

    stop_server(proc)
    time.sleep(5)

print("\ndone. results in /content/results/")
!ls -la /content/results/


=== fp16_mns8 ===
  server up
  concurrency=1
vllm bench serve --backend openai --base-url http://localhost:8000 --model Qwen/Qwen2.5-0.5B-Instruct --num-prompts 100 --max-concurrency 1 --dataset-name random --random-input-len 128 --random-output-len 128 --save-result --result-filename /content/results/fp16_mns8_c1.json
  concurrency=4
vllm bench serve --backend openai --base-url http://localhost:8000 --model Qwen/Qwen2.5-0.5B-Instruct --num-prompts 100 --max-concurrency 4 --dataset-name random --random-input-len 128 --random-output-len 128 --save-result --result-filename /content/results/fp16_mns8_c4.json
  concurrency=16
vllm bench serve --backend openai --base-url http://localhost:8000 --model Qwen/Qwen2.5-0.5B-Instruct --num-prompts 100 --max-concurrency 16 --dataset-name random --random-input-len 128 --random-output-len 128 --save-result --result-filename /content/results/fp16_mns8_c16.json

=== fp16_mns32 ===
  server up
  concurrency=1
vllm bench serve --backend openai --base-u

In [ ]:
import shutil
shutil.make_archive("/content/vllm_sweep_results", "zip", RESULTS_DIR)
from google.colab import files
files.download("/content/vllm_sweep_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>